In [1]:
import pandas as pd 
import requests
import time
import os
from dotenv import load_dotenv
from tqdm import tqdm

In [2]:
# carregando a chave da API do .env

load_dotenv('../.env')
API_KEY = os.getenv('OMDB_API_KEY')

In [3]:
# carregando o csv
# se for rodar pela primeira vez, usa a df_tmdb
# a partir da 2a vez, usa o df_omdb

caminho_omdb = '../data/df_omdb.csv'
caminho_tmdb = '../data/df_tmdb.csv'

if os.path.exists(caminho_omdb):
    df = pd.read_csv(caminho_omdb)
else:
    df = pd.read_csv(caminho_tmdb)

In [4]:
# nova coluna de dados que queremos adicionar no nosso dataset

metascore_critica = []

In [5]:
# limite porque a api limita a 1000 requisições diárias

limite_diario = 950
contador = 0

In [6]:
# coleta de dados do omdb
# para cada linha: se já existe uma nota salva (de um dia anterior), reaproveita;
# senão, se ainda não bateu o limite diário, busca na api; senão, deixa None para tentar no próximo dia

for index, row in tqdm(df.iterrows(), total=len(df)):

    if 'metascore_critica' in df.columns:
        nota_existente = row['metascore_critica']

        if pd.notna(nota_existente):
            metascore_critica.append(nota_existente)
            continue

    if contador >= limite_diario:
        metascore_critica.append(None)
        continue

    imdb_id = row['tconst']
    url_omdb = f"http://www.omdbapi.com/?i={imdb_id}&apikey={API_KEY}"

    try:
        response = requests.get(url_omdb)

        if response.status_code == 200:
            data = response.json()
            score = data.get('Metascore')

            if score and score != "N/A":
                metascore_critica.append(int(score))
            else:
                metascore_critica.append(-1)
        else:
            metascore_critica.append(None)

        contador += 1
        time.sleep(0.05)

    except Exception as e:
        metascore_critica.append(None)

100%|██████████| 12455/12455 [04:21<00:00, 47.60it/s]  


In [7]:
# adicionando a coluna e salvando o progresso

df['metascore_critica'] = metascore_critica
df.to_csv(caminho_omdb, index=False)

faltam = df['metascore_critica'].isna().sum()
print(f'Requisições feitas hoje: {contador}')
print(f'Filmes que ainda faltam buscar: {faltam}')

Requisições feitas hoje: 950
Filmes que ainda faltam buscar: 11505
